<a href="https://colab.research.google.com/github/rachitjbjlkn/CV-builder-CVForge/blob/main/Medical_Diagnoser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("subho117/build-a-deep-learning-based-medical-diagnoser")

100%|██████████| 13.8k/13.8k [00:00<00:00, 7.69MB/s]

Extracting files...


In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Activation, Dense, Dropout, Input, Embedding


In [3]:
print(os.listdir(path))

['medical_data.csv']


In [4]:
data_path = os.path.join(path, 'medical_data.csv')
df = pd.read_csv(data_path)
display(df.head())

,Patient_Problem,Disease,Prescription
0,"Constant fatigue and muscle weakness, struggli...",Chronic Fatigue Syndrome,"Cognitive behavioral therapy, graded exercise ..."
1,"Frequent severe migraines, sensitivity to ligh...",Migraine with Aura,"Prescription triptans, avoid triggers like bri..."
2,"Sudden weight gain and feeling cold, especiall...",Hypothyroidism,Levothyroxine to regulate thyroid hormone levels.
3,"High fever, sore throat, and swollen lymph nod...",Mononucleosis,"Rest and hydration, ibuprofen for pain."
4,"Excessive thirst and frequent urination, dry m...",Diabetes Mellitus,Insulin therapy and lifestyle changes.


In [5]:
df.shape

(407, 3)

In [6]:
tokenizer=Tokenizer(num_words=5000,oov_token="<OOV>")
tokenizer.fit_on_texts(df['Patient_Problem'])
seq=tokenizer.texts_to_sequences(df['Patient_Problem'])


In [7]:
max_len=max(len(i) for i in seq)
padded=pad_sequences(seq,maxlen=max_len,padding='post')

In [8]:
label_disease=LabelEncoder()
label_prescription=LabelEncoder()
disease_labels=label_disease.fit_transform(df['Disease'])
prescription_labels=label_prescription.fit_transform(df['Prescription'])
disease_labels_categoriacl=to_categorical(disease_labels)
prescription_labels_categoriacl=to_categorical(prescription_labels)
disease_labels_categoriacl.shape
prescription_labels_categoriacl.shape

(407, 388)

In [9]:
y=np.hstack(disease_labels_categoriacl)


In [10]:
input_layer=Input(shape=(max_len,))
embedding_layer=Embedding(input_dim=5000,output_dim=100)(input_layer)
lstm_layer=LSTM(128)(embedding_layer)
disease_output=Dense(len(label_disease.classes_),activation='softmax',name='disease_output')(lstm_layer)
prescription_output=Dense(len(label_prescription.classes_),activation='softmax',name='prescription_output')(lstm_layer)

In [11]:
model=Model(inputs=input_layer,outputs=[disease_output,prescription_output])
model.compile(optimizer='adam',loss={'disease_output':'categorical_crossentropy','prescription_output':'categorical_crossentropy'},metrics={'disease_output':['accuracy'],'prescription_output':['accuracy']})
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 17)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 17, 100)   │    500,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 128)       │    117,248 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ disease_output      │ (None, 178)       │     22,962 │ lstm[0][0]        │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prescription_output │ (None, 388)       │     50,052 │ lstm[0][0]        │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 690,262 (2.63 MB)

 Trainable params: 690,262 (2.63 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.fit(padded,{'disease_output':disease_labels_categoriacl,'prescription_output':prescription_labels_categoriacl},epochs=100,batch_size=32,verbose=1)

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - disease_output_accuracy: 0.0098 - disease_output_loss: 5.1802 - loss: 11.1504 - prescription_output_accuracy: 0.0000e+00 - prescription_output_loss: 5.9706
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - disease_output_accuracy: 0.0270 - disease_output_loss: 5.1501 - loss: 11.1164 - prescription_output_accuracy: 0.0074 - prescription_output_loss: 5.9672
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - disease_output_accuracy: 0.0246 - disease_output_loss: 5.0312 - loss: 11.0305 - prescription_output_accuracy: 0.0074 - prescription_output_loss: 6.0046
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - disease_output_accuracy: 0.0319 - disease_output_loss: 4.9208 - loss: 10.9095 - prescription_output_accuracy: 0.0074 - prescription_output_loss: 5.9937
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - disease_output_accuracy: 0.0270 - disease_output_loss: 4.8451 - loss: 10.7838 - prescription_output_accuracy: 0.0147 -

In [13]:
def make_prediction(patient_problem):
  sq=tokenizer.texts_to_sequences([patient_problem])
  pad=pad_sequences(sq,maxlen=max_len,padding='post')
  prediction=model.predict(pad)
  disease_prediction=label_disease.inverse_transform([np.argmax(prediction[0])])
  prescription_prediction=label_prescription.inverse_transform([np.argmax(prediction[1])])
  print(f'Predicted Disease: {disease_prediction[0]}')
  print(f'Suggested Prescription:{prescription_prediction[0]}')

In [16]:
patient_input=input('Enter Patient Problem: ')
make_prediction(patient_input)

Enter Patient Problem: i'm felling dizzing
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Predicted Disease: Type 1 Diabetes
Suggested Prescription:Antiarrhythmic drugs; lifestyle changes.


In [15]:
model.save("rachit_diagnosis.keras")